<img src="https://geo-credito-rural.github.io/_static/logo.jpeg" align="right" width="64" />

# <span style="color:#336699">Uso e Cobertura da Terra</span>

<hr style="border:2px solid #0077b9;">
<div style="text-align: left;">
    <a href="https://nbviewer.jupyter.org/github/brazil-data-cube/code-gallery/blob/master/jupyter/Python/stac/stac-image-processing.ipynb"><img src="https://raw.githubusercontent.com/jupyter/design/master/logos/Badges/nbviewer_badge.svg" align="center"/></a>
</div>

<br/>

<div style="text-align: center;font-size: 90%;">
     Gabriel Sansigolo, Karine Ferreira, Thales Sehn Körting, Gilberto Queiroz, Marcos Adami
    <br/><br/>
    Divisão de Observação da Terra e Geoinformática, Instituto Nacional de Pesquisas Espaciais (INPE)
    <br/>
    Avenida dos Astronautas, 1758, Jardim da Granja, São José dos Campos, SP 12227-010, Brazil
    <br/><br/>
    Contato: <a href="https://geo-credito-rural.github.io/">Equipe - Geo Credito Rural</a>
    <br/><br/>
    Última atualização: 28 de abril de 2026
</div>


<br/>

</div>

Informações sobre uso e cobertura da terra são essenciais para apoiar governos na tomada de decisões sobre o impacto das atividades humanas no meio ambiente, no planejamento do uso dos recursos naturais, na conservação da biodiversidade, no monitoramento das mudanças climáticas e na avaliação da segurança alimentar. O uso da terra é resultado das atividades humanas na superfície terrestre, muitas vezes tendo relação com fatores socioeconômicos, históricos e culturais. Está diretamente associado à cobertura da terra, que vem sendo alterada, e, devido a isso, impacta os ambientes naturais muitas vezes negativamente, moldando-os para atender as demadas da sociedade

### <span style="text-align:center; color:#336699">Instalar e importar os pacotes</span>
<hr style="border:1px solid #0077b9;">

In [ ]:
import subprocess
import sys

def install_package(package_name):
    print(f'\nInstalando {package_name}...')
    proc = subprocess.run([sys.executable, '-m', 'pip', 'install', package_name], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if proc.returncode == 0:
        print(f'{package_name} instalado com sucesso!')
    else:
        print(f'Falha ao instalar {package_name}.')


#Instalando o pacote WLTS
package_wlts = 'git+https://github.com/brazil-data-cube/wlts.py@v1.0.1'

install_package(package_wlts)

In [ ]:
install_package('folium')
install_package('seaborn')
install_package('scikit-learn')

In [ ]:
# Importando todos os pacotes
import os
import io
import wlts
import folium
import shapely
import zipfile
import requests
import warnings
import numpy as np
import pandas as pd
import rasterio as rio
import geopandas as gpd
from rasterio import mask
import earthpy.plot as ep
import matplotlib.pyplot as plt
from IPython.display import display
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from ipywidgets import Accordion, HTML

# Correção de compatibilidade do Numpy
np.bool8 = np.bool_

# Ignorar os avisos "FutureWarning" associados ao pacote WLTS
warnings.simplefilter(action='ignore', category=FutureWarning)

print('Pacotes importados!')

### <span style="text-align:center; color:#336699">Criar o serviço WLTS e recuperar as coleções </span>
<hr style="border:1px solid #0077b9;">

In [ ]:
service = wlts.WLTS('https://data.inpe.br/bdc/wlts/v1/')
service

### <span style="text-align:center; color:#336699">Carregar os dados </span>
<hr style="border:1px solid #0077b9;">

In [ ]:
# Carregar o shapefile da área de estudo

r = requests.get("https://raw.githubusercontent.com/GSansigolo/shapefiles/refs/heads/main/ae.zip")

zipfile.ZipFile(io.BytesIO(r.content)).extractall("./ae.zip")
file_path = "./ae.zip/ae.shp"

shape_area = gpd.read_file(file_path)

In [ ]:
# Carregar o shapefile com as 100 maiores glebas da área de estudo

r = requests.get("https://raw.githubusercontent.com/GSansigolo/shapefiles/refs/heads/main/glebas_selecionadas_100_maiores.zip")

zipfile.ZipFile(io.BytesIO(r.content)).extractall("./glebas_selecionadas_100_maiores.zip")
file_path = "./glebas_selecionadas_100_maiores.zip/glebas_selecionadas_100_maiores.shp"

glebas = gpd.read_file(file_path)

glebas.head()

### <span style="text-align:center; color:#336699"> Plotar as 100 maiores glebas dentro da área de estudo </span>
<hr style="border:1px solid #0077b9;">

In [ ]:
# Plotando a área de estudo apenas com os limites e adicionando uma legenda
ax = shape_area.boundary.plot(edgecolor='black', linewidth=1, figsize=(12, 12), label='Região')

# Plotando as top_100_areas com a cor vermelha
glebas.plot(ax=ax, color='red', label='100 maiores glebas')

# Criando handles personalizados para a legenda
handles = [mpatches.Patch(color='red', label='100 maiores glebas'),
           mpatches.Patch(edgecolor='black', facecolor='none', label='Região')]

plt.title("Plot da área de interesse e as 100 maiores glebas ")
# Adicionando a legenda
plt.legend(handles=handles)

# Exibindo o plot
plt.show()

### <span style="text-align:center; color:#336699"> Verificar o sistema de referência das coordenadas e reprojetar se necessário </span>
<hr style="border:1px solid #0077b9;">

In [ ]:
#Plotar o sistema de referência das coordenadas (CRS - Coordinate Reference System)
glebas.crs

In [ ]:
#A função tj do WLTS espera que o ponto esteja no CRS EPSG:4326

#re-project to another CRS
glebas = glebas.to_crs(4326)

glebas.crs

### <span style="text-align:center; color:#336699"> Escolher uma gleba randomicamente e pegar o seu centroide  </span>
<hr style="border:1px solid #0077b9;">

In [ ]:
# Seleciona randomicamente uma gleba entre as 100 maiores
geometria = glebas.sample(n=1)

# Plota a geometria da gleba
geometria.plot()

In [ ]:
# Calcular o centroide da gleba
centroid_point = geometria.iloc[0]['geometry'].centroid

#Incluir o centroide como uma coluna em geometria
geometria["centroid"] = geometria.iloc[0]["geometry"].centroid

#Plotar o ponto
print(centroid_point)

### <span style="text-align:center; color:#336699"> Plotar a gleba e seu centroide  </span>
<hr style="border:1px solid #0077b9;">

In [ ]:
#Definir o centro do plot
m = folium.Map(location=[float(centroid_point.y), float(centroid_point.x)], zoom_start=17, tiles="CartoDB positron")

#Plotar todas as geometrias e seus centroides

#Plotar as geometrias
for _, r in geometria.iterrows():
    # Without simplifying the representation of each borough,
    # the map might not be displayed
    sim_geo = gpd.GeoSeries(r["geometry"]).simplify(tolerance=0.001)
    geo_j = sim_geo.to_json()
    geo_j = folium.GeoJson(data=geo_j, style_function=lambda x: {"fillColor": "orange"})
    folium.Popup(r["REF_BACEN"]).add_to(geo_j)
    geo_j.add_to(m)

#Plotar os centroides
for _, r in geometria.iterrows():
    lat = r["centroid"].y
    lon = r["centroid"].x
    folium.Marker(
        location=[lat, lon],
        popup="centroid: {} <br>".format(r["centroid"]),
    ).add_to(m)

#Plotar
m

### <span style="text-align:center; color:#336699"> Recuperar e plotar os tipos de uso e cobertura da terra associados à essa localização </span>
<hr style="border:1px solid #0077b9;">

In [ ]:
#Recupera a trajetória de uso e cobertura da terra associada ao ponto

#A função tj espera que o ponto esteja no CRS EPSG:4326

trajectory = service.tj(latitude  = float(centroid_point.y),
                            longitude = float(centroid_point.x),
                            start_date = 2010, end_date = 2024,
                            collections = "mapbiomas-v10")

service.plot(trajectory.df())

### <span style="text-align:center; color:#336699"> Gerar uma grade de pontos dentro da gleba, a cada 30 ou 120 metros </span>
<hr style="border:1px solid #0077b9;">

In [ ]:
# Gerar um ponto a cada 30 metros (1°=111,32 Km - about 0,000269493°)
#x_spacing = 0.0003
#y_spacing = 0.0003

# Gerar um ponto a cada 120 metros (1°=111,32 Km - about 0,0011°)
x_spacing = 0.0011
y_spacing = 0.0011

# Retângulo envolvente (bounding box) do polígono da gleba
xmin, ymin, xmax, ymax = geometria.total_bounds

# Cria a grande de pontos (coordenadas x e y) dentro do retangulo envolvente
xcoords = [c for c in np.arange(xmin, xmax, x_spacing)]
ycoords = [c for c in np.arange(ymin, ymax, y_spacing)]

# Cria os pares x e y
coordinate_pairs = np.array(np.meshgrid(xcoords, ycoords)).T.reshape(-1, 2)

# Cria os pontos
geometries = gpd.points_from_xy(coordinate_pairs[:,0], coordinate_pairs[:,1])

# Cria um GeoDataFrame
pointdf = gpd.GeoDataFrame(geometry=geometries, crs=geometria.crs)

#Pegar somente os pontos que estão dentro do polígono da gleba
pointswithingleba = gpd.sjoin(pointdf, geometria, how='inner', predicate='within')

#Plotar
fig, ax = plt.subplots(figsize=(15, 15))

geometria.plot(ax=ax, alpha=0.7, color="pink")

pointswithingleba.plot(ax=ax)

### <span style="text-align:center; color:#336699"> Recuperar as trajetórias de uso e cobertura da terra desses pontos </span>
<hr style="border:1px solid #0077b9;">

In [ ]:
trajectories = service.tj(
    latitude  = pointswithingleba.geometry.y.to_list(),
    longitude = pointswithingleba.geometry.x.to_list(),
    start_date = 2020,
    end_date = 2022,
    collections = "mapbiomas-v10"
)

### <span style="text-align:center; color:#336699"> Plotar um gráfico de barras com as classes de uso e cobertura da terra que esse ponto foram classificados </span>
<hr style="border:1px solid #0077b9;">

In [ ]:
service.plot(trajectories.df(), type='bar', width=1000, height=500, font_size=15)

In [ ]:
install_package('earthpy')

### <span style="text-align:center; color:#336699"> Carregar os dados </span>
<hr style="border:1px solid #0077b9;">

In [ ]:
# Carregar o shapefile da área de estudo

r = requests.get("https://raw.githubusercontent.com/GSansigolo/shapefiles/refs/heads/main/BR_Pais_2022.zip")

zipfile.ZipFile(io.BytesIO(r.content)).extractall("./BR_Pais_2022.zip")
file_path = "./BR_Pais_2022.zip/BR_Pais_2022.shp"

limites_BR = gpd.read_file(file_path)

In [ ]:
# Carregar o CSV de classes fundiárias

classes = pd.read_csv("https://raw.githubusercontent.com/GSansigolo/shapefiles/refs/heads/main/Codigos-da-legenda-colecao-8.csv", delimiter=';')

In [ ]:
import gdown

file_id = "1n8ZUnGbRpn_e2H8mlGmb2b1dvgk_ihRC"
url = f"https://drive.google.com/uc?id={file_id}"
file_path = "./brasil_coverage_2022.tif"

# Baixar o arquivo do Google Drive
gdown.download(url, file_path, quiet=False)


### <span style="text-align:center; color:#336699"> Plotar a área de estudo </span>
<hr style="border:1px solid #0077b9;">

In [ ]:
# Plotando o shape_area apenas com os limites e adicionando uma legenda
ax = limites_BR.boundary.plot(edgecolor='black', linewidth=1, figsize=(12, 12), label='Região')

# Plotando as top_100_areas com a cor vermelha
shape_area.plot(ax=ax, color='red', label='100 maiores glebas')

# Criando handles personalizados para a legenda
handles = [mpatches.Patch(color='red', label='Região de interesse'),
           mpatches.Patch(edgecolor='black', facecolor='none', label='Limite do Brasil')]

plt.title("Plot  ")
# Adicionando a legenda
plt.legend(handles=handles)

# Exibindo o plot
plt.show()

### <span style="text-align:center; color:#336699"> Plotar as 100 maiores glebas dentro da área de estudo </span>
<hr style="border:1px solid #0077b9;">

In [ ]:
# Plotando a área de estudo apenas com os limites e adicionando uma legenda
ax = shape_area.boundary.plot(edgecolor='black', linewidth=1, figsize=(12, 12), label='Região')

# Plotando as top_100_areas com a cor vermelha
glebas.plot(ax=ax, color='red', label='100 maiores glebas')

# Criando handles personalizados para a legenda
handles = [mpatches.Patch(color='red', label='100 maiores glebas'),
           mpatches.Patch(edgecolor='black', facecolor='none', label='Região')]

plt.title("Plot da área de interesse e as 100 maiores glebas ")
# Adicionando a legenda
plt.legend(handles=handles)

# Exibindo o plot
plt.show()

### <span style="text-align:center; color:#336699"> Filtrar as glebas de 2022, escolher uma gleba randomicamente e cruzar com o raster do MapBiomas de 2022  </span>
<hr style="border:1px solid #0077b9;">

In [ ]:
def recortar_raster_map(geometrias, raster_path):
    with rio.open(raster_path) as src:

        if isinstance(geometrias, shapely.geometry.Polygon):
            geometrias = gpd.GeoDataFrame(geometry=[geometrias], crs='EPSG:4326')
        elif isinstance(geometrias, gpd.GeoSeries):
            geometrias = gpd.GeoDataFrame(geometry=geometrias)

        geometrias = geometrias.to_crs(src.crs)

        shapes = geometrias.geometry.tolist()

        img, transform = mask.mask(src, shapes, crop=True)

        left, bottom, right, top = rio.transform.array_bounds(img.shape[1], img.shape[2], transform)
        xs, ys = rio.warp.transform(src.crs, 'EPSG:4326', [left, right], [bottom, top])

    return img[0], (xs[0], xs[1], ys[0], ys[1])

def plot(img, bbox, classes):
    vals, counts = np.unique(img, return_counts=True)
    cmap_colors, labels = [], []

    valid_mask = vals != -9999
    valid_vals = vals[valid_mask]
    valid_counts = counts[valid_mask]

    for v in vals:
        if v == -9999:
            cmap_colors.append('#000000')
            labels.append("No Data")
        else:
            row = classes[classes['Class_ID'] == v]
            cmap_colors.append(row['Color'].values[0] if not row.empty else '#000000')

            labels.append(row['Descricao'].values[0] if not row.empty else f"Classe {v}")

    cmap = mcolors.ListedColormap(cmap_colors)
    bounds = list(vals) + [vals[-1] + 1]
    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    areas = (valid_counts * 900) / 10000
    total_pixels = valid_counts.sum()

    area_dict = dict(zip(valid_vals, areas))
    pct_dict = dict(zip(valid_vals, [f"{(c / total_pixels * 100):.2f}" for c in valid_counts]))

    df = classes[classes['Class_ID'].isin(valid_vals)].copy()
    df.set_index('Class_ID', inplace=True)

    cols_to_keep = ['Level', 'Descricao']
    cols_to_keep = [col for col in cols_to_keep if col in df.columns]
    df = df[cols_to_keep]

    df['Area (ha)'] = df.index.map(area_dict)
    df['Percentage (%)'] = df.index.map(pct_dict)

    fig, ax = plt.subplots(figsize=(6, 8))
    im = ax.imshow(img, cmap=cmap, norm=norm, extent=bbox, interpolation='none')

    try:
        ep.draw_legend(im, titles=labels)
    except Exception as e:
        print(f"Aviso da Legenda: {e}")

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_title(f'Área Total: {areas.sum():.2f} ha')
    plt.show()

    html_content = f"<div style='background-color:white; padding:10px; border:2px solid black; width:100%;'>{df.to_html()}</div>"
    acc = Accordion(children=[HTML(value=html_content)])
    acc.set_title(0, 'Tabela de Informações')
    display(acc)

In [ ]:
# Filtra as glebas do ano de 2022 garantindo a tipagem de string (evita erro caso seja Datetime)
glebas_2022 = glebas[glebas['DATA_EMISS'].astype(str).str.startswith('2022')]
geometria = glebas_2022.sample(n=1)

# Note: certifique-se de que 'file_path' está definido com o local do seu raster
imagem_cortada, bbox = recortar_raster_map(geometria, file_path)

# Plotar o mapa e mostrar a tabela
plot(imagem_cortada, bbox, classes)